# 02 – Modeling, Ensembles, and Inventory Optimization

This notebook explains the modeling choices, evaluation strategy, ensemble
design, and inventory optimization logic used in **DemandCast**.

It is designed to run in **demo mode** without requiring Kaggle data by
falling back to a small committed demo series under `assets/demo/demo_series.csv`.


## 1. Business objective recap

We focus on forecasting daily demand for a store–item combination and
translating these forecasts into inventory policies:

- **Forecasting goal:** achieve reasonable error on daily demand over a
  medium-term horizon (e.g. 90 days).
- **Inventory goal:** choose order quantities and reorder points that balance
  stockout risk (service level) and holding + ordering cost.


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0]
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from demandcast.data.synthetic import generate_demo_series
from demandcast.features import FeatureConfig, build_time_series_features
from demandcast.evaluation.metrics import rmse, mae, mape, smape, wape
from demandcast.models import (
    ARIMASARIMAXModel,
    ETSModel,
    ProphetFallbackModel,
    LGBMForecaster,
    EnsembleForecaster,
)
from demandcast.inventory import optimize_inventory
from demandcast.paths import DEMO_DATA_DIR
from demandcast.logging_utils import get_logger

logger = get_logger(__name__)


## 2. Demo data loading

We intentionally avoid downloading Kaggle data here. Instead, we use the
committed demo series. If it is missing (for example, in a fresh environment),
we fall back to generating a synthetic series on the fly.

In [2]:
demo_csv = DEMO_DATA_DIR / "demo_series.csv"
if demo_csv.exists():
    series_df = pd.read_csv(demo_csv)
    series_df["ds"] = pd.to_datetime(series_df["ds"])
    series_df = series_df[["ds", "y"]].copy()
    source = f"Demo CSV – {demo_csv}"
else:
    demo = generate_demo_series()
    series_df = demo[["ds", "y"]].copy()
    source = "Synthetic demo series (generated)"

logger.info("Using series source: %s", source)
series_df.head()

2026-01-26 21:36:05,446 | INFO | __main__ | Using series source: Demo CSV – D:\Project\demandcast-inventory-optimizer\assets\demo\demo_series.csv


,ds,y
0,2017-01-01,18.021757
1,2017-01-02,23.183584
2,2017-01-03,27.470490
3,2017-01-04,22.587368
4,2017-01-05,19.711043


## 3. Modeling approach overview

We consider a small ensemble of diverse models:

- **SARIMAX** – classical linear time-series model for trend/seasonality.
- **ETS** – Exponential Smoothing for level/trend/seasonal components.
- **Prophet-or-fallback** – Prophet when installed, SARIMAX otherwise.
- **LightGBM** – gradient boosting on engineered lag/rolling/calendar features.

The `EnsembleForecaster` blends these models using weights derived from
backtest performance (e.g. inverse WAPE) and exposes conservative prediction
intervals that take the min of lower bounds and the max of upper bounds
across components.


## 4. Feature engineering

`FeatureConfig` and `build_time_series_features` create a feature matrix from
the univariate series by adding:

- calendar features (day-of-week, month, etc.)
- lags (e.g. 1, 7, 14, 28 days)
- rolling window statistics (mean and std over weekly / monthly windows)

These features feed the LightGBM model and any other tabular learners.

In [3]:
features_cfg = FeatureConfig()
features_df = build_time_series_features(series_df, config=features_cfg)
features_df.head()

2026-01-26 21:36:05,469 | INFO | demandcast.features.build_features | Built time series features: 730 rows, 15 columns


,ds,y,dow,doy,week,month,year,y_lag_1,y_lag_7,y_lag_14,y_lag_28,y_roll_mean_7,y_roll_std_7,y_roll_mean_28,y_roll_std_28
0,2017-01-01,18.021757,6,1,52,1,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-01-02,23.183584,0,2,1,1,2017,18.021757,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017-01-03,27.470490,1,3,1,1,2017,23.183584,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-01-04,22.587368,2,4,1,1,2017,27.470490,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-01-05,19.711043,3,5,1,1,2017,22.587368,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Backtesting methodology

The project uses **rolling-origin backtesting** with an expanding training
window:

1. Train on an initial history window.
2. Forecast a fixed horizon.
3. Slide the window forward and repeat.

This approximates real deployment: the model is always trained on past data
and evaluated on unseen future periods. Metrics are aggregated across folds.


## 6. Metrics and their roles

We track several metrics:

- **RMSE** – penalizes large errors, kept on the original scale.
- **MAE** – robust average absolute error.
- **MAPE** – percentage error, but unstable near zero.
- **sMAPE** – symmetric version of MAPE, more robust when values are small.
- **WAPE** – weighted absolute percentage error, robust to scale and useful
  in inventory contexts.

For inventory decisions, **WAPE** is particularly intuitive: it approximates
how much total demand volume we mis-forecast as a fraction of total demand.


## 7. Interval estimation and conformal prediction

Classical models like SARIMAX and ETS expose analytical or approximate
prediction intervals (e.g. via `get_forecast` in statsmodels). For LightGBM,
we:

1. Train a base `LGBMRegressor` on engineered features.
2. Wrap it in a `MapieTimeSeriesRegressor` configured with the EnbPI
   (Ensemble Bootstrap Prediction Interval) method.
3. Use residual-based conformal prediction to produce intervals that aim for
   the desired coverage without relying on parametric distributional
   assumptions.

Coverage and interval width are then summarized per-fold and overall.


## 8. LightGBM feature importance

Two complementary notions of feature importance are tracked:

1. **Gain importance** (from LightGBM itself):
   - Measures the total reduction in loss contributed by splits that use
     each feature.
   - Good for understanding which features the model relies on globally.
2. **Permutation importance**:
   - Randomly permutes one feature at a time on a validation set.
   - Measures how much the metric degrades when that feature is scrambled.
   - More directly tied to predictive performance and robust to some model
     biases.

Both sets of importances are written to CSV files in `artifacts/reports/` by
`scripts/run_step3_train_lgbm.py`.


## 9. Ensemble weighting

The `EnsembleForecaster` assigns non-negative weights to each base model
(SARIMAX, ETS, Prophet-or-fallback, LightGBM) and normalizes them to sum to
1. The default strategy uses **inverse WAPE** from the backtest:

- Models with lower WAPE receive higher weight.
- Models with missing or invalid metrics receive weight 0.

Point forecasts are a weighted average of the component predictions. For
intervals, the default **conservative** scheme takes the minimum lower bound
and maximum upper bound across components, avoiding overly narrow bands when
models disagree.


## 10. Inventory optimization: EOQ, ROP, and safety stock

Given a forecast for daily demand, DemandCast derives inventory policies using
classical formulas:

- **EOQ (Economic Order Quantity)**
  - \( Q^* = \sqrt{2 D S / H} \)
  - \( D \): annual demand, \( S \): ordering cost, \( H \): holding cost per
    unit per year.
- **Reorder point (ROP)**
  - Mean demand during lead time plus safety stock.
- **Safety stock** (demand-only variability)
  - \( SS = z \cdot \sigma_d \sqrt{L} \)
  - \( \sigma_d \): daily demand std, \( L \): lead time in days, \( z \):
    z-score corresponding to the target service level.

Annual ordering and holding costs are computed and combined into a total
annual cost used to compare policies, including a naïve baseline (e.g. fixed
monthly orders without safety stock).


In [4]:
# Example: build a simple forecast from the demo series using the ensemble

ensemble = EnsembleForecaster()
ensemble.fit(series_df)

last_ds = series_df["ds"].max()
horizon = 30
future_ds = pd.date_range(start=last_ds + pd.Timedelta(days=1), periods=horizon, freq="D")
future_df = pd.DataFrame({"ds": future_ds})
forecast_df = ensemble.predict(future_df, horizon=horizon)
forecast_df.head()

2026-01-26 21:36:08,977 | INFO | demandcast.models.ensemble | Fitting ensemble base models on 730 observations.
2026-01-26 21:36:08,978 | INFO | demandcast.models.prophet_fallback | Prophet is available; ProphetFallbackModel will use Prophet.
2026-01-26 21:36:08,978 | INFO | demandcast.models.ensemble | Fitting base model 'sarimax' for ensemble.
2026-01-26 21:36:08,979 | INFO | demandcast.models.arima_sarimax | Fitting SARIMAX model with order=(1, 0, 0), seasonal_order=(0, 0, 0, 0) on 730 observations
2026-01-26 21:36:09,044 | INFO | demandcast.models.ensemble | Fitting base model 'ets' for ensemble.
2026-01-26 21:36:09,046 | INFO | demandcast.models.ets | Fitting ETS model with trend=add, seasonal=add, seasonal_periods=7
2026-01-26 21:36:09,211 | INFO | demandcast.models.ensemble | Fitting base model 'prophet_or_fallback' for ensemble.
2026-01-26 21:36:09,212 | INFO | demandcast.models.prophet_fallback | Fitting Prophet model on 730 observations.
21:36:09 - cmdstanpy - INFO - Chain [1

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000275 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2191
[LightGBM] [Info] Number of data points in the train set: 702, number of used features: 13
[LightGBM] [Info] Start training from score 23.770101
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

2026-01-26 21:36:09,688 | INFO | demandcast.features.build_features | Built time series features: 733 rows, 15 columns
2026-01-26 21:36:09,700 | INFO | demandcast.features.build_features | Built time series features: 734 rows, 15 columns
2026-01-26 21:36:09,712 | INFO | demandcast.features.build_features | Built time series features: 735 rows, 15 columns
2026-01-26 21:36:09,724 | INFO | demandcast.features.build_features | Built time series features: 736 rows, 15 columns
2026-01-26 21:36:09,736 | INFO | demandcast.features.build_features | Built time series features: 737 rows, 15 columns
2026-01-26 21:36:09,749 | INFO | demandcast.features.build_features | Built time series features: 738 rows, 15 columns
2026-01-26 21:36:09,761 | INFO | demandcast.features.build_features | Built time series features: 739 rows, 15 columns
2026-01-26 21:36:09,773 | INFO | demandcast.features.build_features | Built time series features: 740 rows, 15 columns
2026-01-26 21:36:09,785 | INFO | demandcast.feat

,ds,yhat,yhat_lower,yhat_upper
0,2019-01-01,32.284645,11.895463,52.537149
1,2019-01-02,30.445794,8.902633,49.544319
2,2019-01-03,27.662049,4.707166,45.348851
3,2019-01-04,24.774466,2.027421,44.498795
4,2019-01-05,24.603847,2.596358,45.517366


## 11. Scenario analysis – conservative, balanced, aggressive

The inventory module defines three presets:

- **Conservative** – higher service level and extra lead-time buffer.
- **Balanced** – mid-range service level suited to many SKUs.
- **Aggressive** – lower service level and shorter effective lead time.

`optimize_inventory` computes an optimized policy and baseline comparison,
then evaluates all three scenario presets. The results are written to
`artifacts/reports/inventory_recommendations.csv` and
`artifacts/reports/inventory_scenarios.csv` by
`scripts/run_step4_inventory_planning.py`.


In [5]:
result = optimize_inventory(
    forecast_df,
    lead_time_days=7,
    order_cost=50.0,
    holding_cost_pct=0.2,
    unit_cost=10.0,
    service_level=0.95,
)
result.keys()

2026-01-26 21:36:11,142 | INFO | demandcast.inventory.optimizer | Forecast summary: avg_daily=27.177, std_daily=2.483, annual=9919.647
2026-01-26 21:36:11,143 | INFO | demandcast.inventory.policies | Built inventory policy: {'order_quantity': 704.2601358031346, 'reorder_point': 201.04370124315676, 'safety_stock': 10.80390002835414, 'expected_cycle_stock': 352.1300679015673, 'days_of_cover': 25.913720047760183, 'annual_demand': 9919.646777628994, 'eoq': 704.2601358031346, 'ordering_cost_annual': 704.2601358031347, 'holding_cost_annual': 725.8679358598429, 'total_cost_annual': 1430.1280716629776}
2026-01-26 21:36:11,144 | INFO | demandcast.inventory.policies | Built inventory policy: {'order_quantity': 704.2601358031346, 'reorder_point': 243.06493144470352, 'safety_stock': 14.77716998694037, 'expected_cycle_stock': 352.1300679015673, 'days_of_cover': 25.913720047760183, 'annual_demand': 9919.646777628994, 'eoq': 704.2601358031346, 'ordering_cost_annual': 704.2601358031347, 'holding_cost_

dict_keys(['inputs', 'optimized_policy', 'baseline_policy', 'cost_improvement_pct_vs_baseline', 'scenarios'])

The `optimized_policy`, `baseline_policy`, and list of `scenarios` can be
inspected directly within this notebook or consumed by the Streamlit app.

## 12. Final recommendations and limitations

This reference implementation is intentionally simple and educational:

- Forecasting models are reasonably strong baselines but not heavily tuned.
- LightGBM features are generic; domain-specific features (price, promos,
  holidays) could significantly improve performance.
- Inventory formulas assume independent periods and approximate normal
  demand; real-world demand may be intermittent or highly skewed.
- Scenario presets are illustrative; real deployments should calibrate
  service levels and cost parameters with business stakeholders.

Nonetheless, the full workflow – from data to forecasts to inventory
policies, with backtests and a Streamlit dashboard – provides a solid
template for experimentation and teaching.
